# `fig3_01_real_panels` — Figure 3 panels from a FelixGenome run

Writes the observed-data files `manuscript_figures/fig3_sim.py` loads in place of
its simulated placeholders. Run **on the Terra Workbench VM**, after a
FelixGenome run; `felix_02_genome_review` answers whether the run worked, this
one turns it into figure inputs.

| File written into the manuscript repo's `data/` | Replaces | Needs |
|---|---|---|
| `fig3_ancestry_vs_pooled.tsv.gz` | `sim_ancestry_vs_pooled` | any FelixGenome run |
| `fig3_dosage_vs_biallelic.tsv.gz` | `sim_dosage_vs_biallelic` | `ru_biallelic_arm = true` |
| `fig3_locus_cards.tsv` | `card_effects` | a chosen locus list |
| `fig3_hits.tsv` | `build_hits` (panel A colours, scan panel) | the genome-wide scan |

A panel falls back to simulation whenever its file is absent, so partial output
is fine and layout work keeps working. `FIG3_SIM_ONLY=1` forces simulation even
when the files exist.

Each file is written with the columns `fig3_sim` expects; nothing here renders
the figure.

## Config

In [ ]:
import os
import re
import subprocess
from pathlib import Path

RUN_ROOT = os.environ.get("FELIX_RUN_ROOT", "").rstrip("/")
if not RUN_ROOT:
    bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
    sub, wf = os.environ.get("FELIX_SUBMISSION_ID", ""), os.environ.get("FELIX_WORKFLOW_ID", "")
    if bucket and sub and wf:
        RUN_ROOT = f"{bucket}/submissions/{sub}/FelixGenome/{wf}"
if not RUN_ROOT.startswith("gs://"):
    raise ValueError("set FELIX_RUN_ROOT, or FELIX_SUBMISSION_ID + FELIX_WORKFLOW_ID with WORKSPACE_BUCKET")

PHENOTYPES = [p for p in os.environ.get("FELIX_PHENOTYPES", "").split(",") if p]
if not PHENOTYPES:
    raise ValueError("set FELIX_PHENOTYPES to the selected_phenotypes order (shards follow it)")

# Where fig3_sim.py looks: manuscript_figures/style.py ROOT / "data".
DATA_DIR = Path(os.environ.get("FIG3_DATA_DIR", "")) if os.environ.get("FIG3_DATA_DIR") else None
if DATA_DIR is None:
    try:
        from manuscript_figures.style import ROOT as FIG_ROOT
        DATA_DIR = Path(FIG_ROOT) / "data"
    except Exception:
        DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

WORK = Path(os.environ.get("FIG3_WORK_DIR", "fig3_inputs")).resolve()
WORK.mkdir(parents=True, exist_ok=True)

# Columns in the FELIX result TSVs.
P_POOLED, P_CCT, P_HET = "p.value_c_ancALL", "P_cct_admixed_c", "P_het_admixed_c"
GW_P = 5e-8

print("run root:", RUN_ROOT)
print("data dir:", DATA_DIR)
print("phenotypes:", len(PHENOTYPES))


def gs_ls(pattern):
    try:
        out = subprocess.run(["gsutil", "ls", pattern], check=True, capture_output=True, text=True).stdout
    except (subprocess.CalledProcessError, FileNotFoundError):
        return []
    return [u for u in out.split() if u.startswith("gs://")]


def shard_uris(call, name="merged.felix.tsv"):
    """Per-phenotype merged TSVs for a Concat-style call, ordered by shard index."""
    uris = gs_ls(f"{RUN_ROOT}/call-{call}/shard-*/**/{name}")
    def idx(u):
        m = re.search(r"/shard-(\d+)/", u)
        return int(m.group(1)) if m else -1
    return [u for _, u in sorted((idx(u), u) for u in uris)]


## 1. Pull the columns each panel needs

The per-phenotype scans are tens of GB genome-wide, so this never downloads
them: `gsutil cat` streams each one through `cut`, which keeps only the handful
of columns the panels use.

In [ ]:
import gzip
import pandas as pd

def stream_columns(uri, columns, out_path):
    """Stream a results TSV from GCS and keep only `columns`, gzipped."""
    header = subprocess.run(f"gsutil cat {uri} | head -1", shell=True,
                            check=True, capture_output=True, text=True).stdout.rstrip("\n").split("\t")
    missing = [c for c in columns if c not in header]
    if missing:
        raise KeyError(f"{uri} lacks columns {missing}")
    idx = ",".join(str(header.index(c) + 1) for c in columns)
    cmd = f"gsutil cat {uri} | cut -f{idx} | gzip -c > {out_path}"
    subprocess.run(cmd, shell=True, check=True)
    return out_path


def load_arm(call, columns, tag):
    """phenotype -> DataFrame of the requested columns, cached under WORK."""
    uris = shard_uris(call)
    if not uris:
        print(f"{call}: no results found; skipping")
        return {}
    out = {}
    for i, uri in enumerate(uris):
        pheno = PHENOTYPES[i] if i < len(PHENOTYPES) else f"shard{i}"
        cache = WORK / f"{tag}_{pheno}.tsv.gz"
        if not cache.is_file():
            stream_columns(uri, columns, cache)
        out[pheno] = pd.read_csv(cache, sep="\t", low_memory=False)
        print(f"  {tag} {pheno}: {len(out[pheno]):,} rows")
    return out


KEYS = ["CHR", "POS", "MarkerID"]
ru = load_arm("RuConcat", KEYS + [P_POOLED, P_CCT, P_HET], "ru")
main = load_arm("Concat", KEYS + [P_POOLED, P_CCT, P_HET], "main")
ba = load_arm("BaConcat", KEYS + [P_POOLED, P_CCT, P_HET], "ba")


## 2. `fig3_ancestry_vs_pooled.tsv.gz`

Pooled test vs the ancestry-aware Cauchy combination, with the heterogeneity
p-value the panel uses to colour points. Built from the repeat arm when it ran,
otherwise the per-record scan.

In [ ]:
import numpy as np

source = ru if ru else main
SOURCE_NAME = "repeat dosage" if ru else "per-record scan"
if not source:
    print("no results; skipping")
else:
    frames = []
    for pheno, df in source.items():
        cols = {"p_pooled": P_POOLED, "p_ancestry_cct": P_CCT, "p_het": P_HET}
        vals = {k: pd.to_numeric(df[v], errors="coerce") for k, v in cols.items()}
        ok = pd.concat(vals.values(), axis=1).notna().all(axis=1)
        frames.append(pd.DataFrame({
            "phenotype": pheno, "locus": df.loc[ok, "MarkerID"].values,
            "chrom": df.loc[ok, "CHR"].values, "pos": df.loc[ok, "POS"].values,
            **{k: v[ok].values for k, v in vals.items()},
        }))
    anc = pd.concat(frames, ignore_index=True)
    out = DATA_DIR / "fig3_ancestry_vs_pooled.tsv.gz"
    anc.to_csv(out, sep="\t", index=False)

    d = -np.log10(anc.p_ancestry_cct.clip(lower=1e-300)) + np.log10(anc.p_pooled.clip(lower=1e-300))
    print(f"wrote {out}  ({len(anc):,} rows from the {SOURCE_NAME})")
    print(f"  above the diagonal by >1 log10 unit: {int((d > 1).sum()):,}; below: {int((d < -1).sum()):,}")
    print(f"  ancestry-aware significant where pooled is not: "
          f"{int(((anc.p_ancestry_cct < GW_P) & (anc.p_pooled >= GW_P)).sum())}")
    print(f"  heterogeneous (P_het < 1e-3): {int((anc.p_het < 1e-3).sum()):,}")


## 3. `fig3_dosage_vs_biallelic.tsv.gz`

The same loci under the two codings, joined per phenotype on the locus id. Needs
`ru_biallelic_arm = true`; the panel stays simulated otherwise.

In [ ]:
if not (ru and ba):
    print("need both the repeat and biallelic arms; the dosage panel stays simulated")
else:
    frames = []
    for pheno in ru:
        if pheno not in ba:
            continue
        a = ru[pheno][["MarkerID", "CHR", "POS", P_CCT]].rename(columns={P_CCT: "p_dosage"})
        b = ba[pheno][["MarkerID", P_CCT]].rename(columns={P_CCT: "p_biallelic"})
        j = a.merge(b, on="MarkerID", how="inner")
        j["p_dosage"] = pd.to_numeric(j.p_dosage, errors="coerce")
        j["p_biallelic"] = pd.to_numeric(j.p_biallelic, errors="coerce")
        j = j.dropna(subset=["p_dosage", "p_biallelic"])
        j.insert(0, "phenotype", pheno)
        frames.append(j.rename(columns={"MarkerID": "locus", "CHR": "chrom", "POS": "pos"}))
        print(f"  {pheno}: {len(j):,} loci in both arms")
    dos = pd.concat(frames, ignore_index=True)
    out = DATA_DIR / "fig3_dosage_vs_biallelic.tsv.gz"
    dos[["phenotype", "locus", "chrom", "pos", "p_biallelic", "p_dosage"]].to_csv(out, sep="\t", index=False)
    gain = (dos.p_dosage < GW_P) & (dos.p_biallelic >= GW_P)
    print(f"wrote {out}  ({len(dos):,} rows)")
    print(f"  significant with dosage only: {int(gain.sum())}; with biallelic only: "
          f"{int(((dos.p_biallelic < GW_P) & (dos.p_dosage >= GW_P)).sum())}")


## 4. `fig3_hits.tsv`

Genome-wide significant associations, which colour panel A's cells and fill the
scan panel. One row per phenotype × locus crossing 5e-8 in either arm.

In [ ]:
hit_frames = []
for arm_name, arm in (("per-record", main), ("repeat dosage", ru), ("repeat biallelic", ba)):
    for pheno, df in arm.items():
        p = pd.to_numeric(df[P_CCT], errors="coerce")
        sel = p < GW_P
        if sel.any():
            hit_frames.append(pd.DataFrame({
                "arm": arm_name, "phenotype": pheno,
                "locus": df.loc[sel, "MarkerID"].values,
                "chrom": df.loc[sel, "CHR"].values, "pos": df.loc[sel, "POS"].values,
                "p": p[sel].values,
            }))
if hit_frames:
    hits = pd.concat(hit_frames, ignore_index=True).sort_values("p")
    out = DATA_DIR / "fig3_hits.tsv"
    hits.to_csv(out, sep="\t", index=False)
    print(f"wrote {out}  ({len(hits)} associations)")
    display(hits.head(20))
else:
    print(f"no association reaches {GW_P:g}; panel A colours and the scan panel stay simulated")
    strongest = []
    for arm_name, arm in (("per-record", main), ("repeat dosage", ru)):
        for pheno, df in arm.items():
            p = pd.to_numeric(df[P_CCT], errors="coerce")
            if p.notna().any():
                i = p.idxmin()
                strongest.append({"arm": arm_name, "phenotype": pheno,
                                  "locus": df.loc[i, "MarkerID"], "p": p[i]})
    if strongest:
        print("\nstrongest per phenotype and arm:")
        display(pd.DataFrame(strongest).sort_values("p").head(12))


## 5. `fig3_locus_cards.tsv`

Per-ancestry effect and standard error at the loci the cards show. Set
`CARD_LOCI` to the locus ids you want; the defaults are the strongest
associations found above, which is only a placeholder until there are real
hits.

In [ ]:
CARD_LOCI = [l for l in os.environ.get("FIG3_CARD_LOCI", "").split(",") if l]
if not CARD_LOCI:
    pool = ru if ru else main
    best = []
    for pheno, df in pool.items():
        p = pd.to_numeric(df[P_CCT], errors="coerce")
        if p.notna().any():
            best.append((p.min(), df.loc[p.idxmin(), "MarkerID"]))
    CARD_LOCI = [m for _, m in sorted(best)[:4]]
print("card loci:", CARD_LOCI)

ANC = [f"anc{k}" for k in range(1, 6)]
CARD_COLS = ["MarkerID", "CHR", "POS"] + [f"{s}_{a}" for a in ANC for s in ("BETA_c", "SE_c", "p.value_c")]

if CARD_LOCI:
    frames = []
    for arm_name, call, tag in (("repeat dosage", "RuConcat", "ru_cards"), ("per-record", "Concat", "main_cards")):
        uris = shard_uris(call)
        for i, uri in enumerate(uris):
            pheno = PHENOTYPES[i] if i < len(PHENOTYPES) else f"shard{i}"
            cache = WORK / f"{tag}_{pheno}.tsv.gz"
            if not cache.is_file():
                try:
                    stream_columns(uri, CARD_COLS, cache)
                except KeyError as e:
                    print(f"  {arm_name} {pheno}: {e}"); continue
            df = pd.read_csv(cache, sep="\t", low_memory=False)
            sel = df[df.MarkerID.isin(CARD_LOCI)]
            if len(sel):
                sel = sel.copy(); sel.insert(0, "phenotype", pheno); sel.insert(0, "arm", arm_name)
                frames.append(sel)
    if frames:
        cards = pd.concat(frames, ignore_index=True)
        out = DATA_DIR / "fig3_locus_cards.tsv"
        cards.to_csv(out, sep="\t", index=False)
        print(f"wrote {out}  ({len(cards)} rows)")
        display(cards.head(12))
    else:
        print("no card rows matched; the card panel stays simulated")


## 6. What is now observed

Re-import `fig3_sim` and read back which panels will draw real data.

In [ ]:
import importlib
import sys

for name in [m for m in list(sys.modules) if m.startswith("manuscript_figures")]:
    del sys.modules[name]
try:
    from manuscript_figures import fig3_sim as S
    print("panel data sources")
    for key in S.REAL_FILES:
        print(f"  {key:9s} {S.panel_tag(key):9s} {S.real_path(key) or '(simulated)'}")
    print("\nset FIG3_SIM_ONLY=1 to force simulation while working on layout")
except Exception as e:
    print("fig3_sim not importable here:", e)
    for f in sorted(DATA_DIR.glob("fig3_*")):
        print("  wrote", f.name)
